# 代理服务器
使用 FastMCP 充当中介或更改其他 MCP 服务器的传输。

FastMCP 提供了强大的代理功能，允许一个 FastMCP 服务器实例作为另一个 MCP 服务器（可以是远程的、运行在不同的传输协议上，甚至是另一个 FastMCP 实例）的前端。这是通过`FastMCP.as_proxy()`类方法实现的。

`as_proxy()`接受现有的Client或任何可以传递给Client其transport参数的参数 - 例如另一个FastMCP实例或远程服务器的 `URL`。


## 什么是代理？
代理是指设置一个 `FastMCP` 服务器，该服务器不直接实现自身的工具或资源。相反，当它收到请求（例如`tools/call`或`resources/read`）时，它会将该请求转发到后端MCP 服务器，接收响应，然后将该响应转发回原始客户端。

## 用例
- 传输桥接：通过不同的传输（例如，Claude Desktop 的本地 Stdio）公开在一种传输上运行的服务器（例如，远程 SSE 服务器）。
- 添加功能：在现有服务器前面插入一个层，以添加缓存、日志记录、身份验证或修改请求/响应（尽管直接修改需要子类化FastMCPProxy）。
- 安全边界：使用代理作为内部服务器的受控网关。
- 简化客户端配置：即使后端服务器的位置或传输发生变化，也能提供单一、稳定的端点（代理）。

## 创建代理
创建代理最简单的方法是使用`FastMCP.as_proxy()`类方法。这将创建一个标准的 FastMCP 服务器，并将请求转发到另一个 MCP 服务器。

In [ ]:
from fastmcp import FastMCP

# Provide the backend in any form accepted by Client
proxy_server = FastMCP.as_proxy(
    "backend_server.py",  # Could also be a FastMCP instance or a remote URL
    name="MyProxyServer"  # Optional settings for the proxy
)

# Or create the Client yourself for custom configuration
backend_client = Client("backend_server.py")
proxy_from_client = FastMCP.as_proxy(backend_client)

`as_proxy`工作原理：

- 它使用提供的客户端连接到后端服务器。
- 它发现后端服务器上可用的所有工具、资源、资源模板和提示。
- 它创建相应的“代理”组件，将请求转发到后端。
- 它返回一个FastMCP可以像其他任何实例一样使用的标准服务器实例。

## 桥接传输
一个常见的用例是桥接传输。例如，通过 Stdio 使远程 SSE 服务器在本地可用：

In [ ]:
from fastmcp import FastMCP

# Target a remote SSE server directly by URL
proxy = FastMCP.as_proxy("http://example.com/mcp/sse", name="SSE to Stdio Proxy")

# The proxy can now be used with any transport
# No special handling needed - it works like any FastMCP server

## In-Memory 代理

您还可以代理内存FastMCP实例，这对于调整您无法完全控制的服务器的配置或行为很有用。

In [ ]:
from fastmcp import FastMCP

# Original server
original_server = FastMCP(name="Original")

@original_server.tool()
def tool_a() -> str: 
    return "A"

# Create a proxy of the original server directly
proxy = FastMCP.as_proxy(
    original_server,
    name="Proxy Server"
)

# proxy is now a regular FastMCP server that forwards
# requests to original_server

## FastMCPProxy 类
在内部`FastMCP.as_proxy()`使用`FastMCPProxy`该类。通常情况下，您不需要直接与此类交互，但如果需要，可以使用它。

对于高级场景，可能需要直接使用该类，例如`FastMCPProxy`在转发请求之前或之后通过子类化来添加自定义逻辑。

